<a href="https://colab.research.google.com/github/SintiaPlasencia/Fitness-Plan-Builder/blob/main/notebooks/Fitness_Plan_Builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

install

In [1]:
!pip install -q transformers accelerate torch huggingface_hub

add API key

In [2]:
from huggingface_hub import login
import getpass

hf_token = getpass.getpass("Paste your Hugging Face token: ")
login(hf_token)

Paste your Hugging Face token: ··········


load llama model

In [3]:
from transformers import pipeline
import torch

generator = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

fitness generator

In [6]:
def generate_fitness_plan(goal, level, days_per_week, workout_type, session_length, notes=""):
    messages = [
        {
            "role": "system",
            "content": """
You are a helpful fitness planning assistant.
Create safe, realistic, structured workout plans.
Do not repeat sentences.
Do not add extra sections.
Use simple and clear formatting.
"""
        },
        {
            "role": "user",
            "content": f"""
Create a personalized fitness plan using EXACTLY this format:

1. Summary
Write 2 short sentences about the user's goal.

2. Weekly Schedule
Create exactly {days_per_week} workout days.
Each day should have a focus area.

3. Exercises with Sets and Reps
For each day, list 3 to 5 exercises.
Include sets, reps, and rest time.

4. Tips
Give 3 practical tips.

User Details:
Goal: {goal}
Experience Level: {level}
Workout Days Per Week: {days_per_week}
Workout Type: {workout_type}
Session Length: {session_length} minutes
Additional Notes: {notes}

Rules:
- Keep it realistic for the user's experience level.
- Do not repeat yourself.
- Do not include medical advice.
- Keep the answer organized and concise.
"""
        }
    ]

    result = generator(
        messages,
        max_new_tokens=500,
        do_sample=False,
        return_full_text=False
    )

    return result[0]["generated_text"]

interactive demo

In [7]:
plan = generate_fitness_plan(
    goal="Weight Loss",
    level="Beginner",
    days_per_week="3",
    workout_type="Gym",
    session_length="45",
    notes="No injuries"
)

print(plan)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. Summary
The user aims to lose weight through a structured gym workout plan, focusing on a beginner-friendly routine that promotes overall fitness and weight loss.

2. Weekly Schedule
- Monday: Upper Body Focus
- Wednesday: Lower Body Focus
- Friday: Cardio and Core Focus

3. Exercises with Sets and Reps
**Monday (Upper Body Focus)**
- Bench Press (3 sets of 8-12 reps, 60 seconds rest)
- Incline Dumbbell Press (3 sets of 10-15 reps, 60 seconds rest)
- Bent Over Rows (3 sets of 8-12 reps, 60 seconds rest)
- Bicep Curls (3 sets of 12-15 reps, 60 seconds rest)
- Tricep Dips (3 sets of 12-15 reps, 60 seconds rest)

**Wednesday (Lower Body Focus)**
- Squats (3 sets of 8-12 reps, 60 seconds rest)
- Leg Press (3 sets of 10-15 reps, 60 seconds rest)
- Lunges (3 sets of 10-15 reps (per leg), 60 seconds rest)
- Calf Raises (3 sets of 12-15 reps, 60 seconds rest)
- Leg Extensions (3 sets of 12-15 reps, 60 seconds rest)

**Friday (Cardio and Core Focus)**
- Treadmill or Stationary Bike (20-30 mi